In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [3]:
import pandas as pd
import polars as pl

assay_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "head-to-head")
    .filter(pl.col("measurand").str.starts_with("beats:"))
    .rename({"value": "score"})
    .with_columns(
        pl.col("measurand").str.replace("^beats:", "").alias("right_entity_id")
    )
    .rename(
        {
            "entity_id": "left_entity_id",
            "entity_name": "left_entity_name",
        }
    )
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "left_entity_id",
        "left_entity_name",
        "right_entity_id",
    )
)

pdf = assay_df.to_pandas()

for col in [
    "model",
    "left_entity_id",
    "right_entity_id",
    "comparison_set_id",
    "assay_instance_hash",
]:
    pdf[col] = pdf[col].astype("category")

pdf["score"] = pd.to_numeric(pdf["score"])

print(pdf.dtypes)

score                   float64
assay                       str
assay_instance_hash    category
sample_number            uint64
model                  category
comparison_set_id      category
comparison_set_name         str
left_entity_id         category
left_entity_name            str
right_entity_id        category
dtype: object


In [4]:
%%R -i pdf

df <- pdf

df$model <- factor(df$model)
df$comparison_set_id <- factor(df$comparison_set_id)
df$assay_instance_hash <- factor(df$assay_instance_hash)

# Make left/right use the same entity level universe
entity_levels <- sort(unique(c(
  as.character(df$left_entity_id),
  as.character(df$right_entity_id)
)))

df$left_entity_id <- factor(df$left_entity_id, levels = entity_levels)
df$right_entity_id <- factor(df$right_entity_id, levels = entity_levels)

# Sum-code only top-level direct factors
contrasts(df$model) <- contr.sum(nlevels(df$model))
contrasts(df$comparison_set_id) <- contr.sum(nlevels(df$comparison_set_id))


make_local_pairwise_sum_contrasts <- function(data, group_var, left_var, right_var) {
  group <- data[[group_var]]
  left <- data[[left_var]]
  right <- data[[right_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g

    kids <- sort(unique(c(
      as.character(left[idx]),
      as.character(right[idx])
    )))

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M_left <- matrix(0, nrow = nrow(data), ncol = ncol(C))
    M_right <- matrix(0, nrow = nrow(data), ncol = ncol(C))

    M_left[idx, ] <- C[as.character(left[idx]), , drop = FALSE]
    M_right[idx, ] <- C[as.character(right[idx]), , drop = FALSE]

    M <- M_left - M_right

    colnames(M) <- paste0(
      "D_", left_var, "_minus_", right_var,
      "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}


make_local_sum_contrasts <- function(data, group_var, child_var) {
  group <- data[[group_var]]
  child <- data[[child_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g
    kids <- sort(unique(as.character(child[idx])))

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M <- matrix(0, nrow = nrow(data), ncol = ncol(C))
    M[idx, ] <- C[as.character(child[idx]), , drop = FALSE]

    colnames(M) <- paste0(
      child_var, "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}


D_E_within_set <- make_local_pairwise_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  left_var = "left_entity_id",
  right_var = "right_entity_id"
)

A_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "assay_instance_hash"
)

fit <- lm(
  score ~
    model * comparison_set_id +
    D_E_within_set +
    A_within_set +
    model:D_E_within_set,
  data = df
)

/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "assay_instance_hash". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "sample_number". Fall back to string conversion. The error is: Cannot convert numpy array of <class 'numpy.uint64'> (R integers are signed 32-bit integers).
  warnings.warn('Error while trying to convert '


In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages


In [5]:
%%R

is_sum_coded <- function(x, name, tol = 1e-8) {
  cm <- contrasts(x)

  ok_dim <- all(dim(cm) == c(nlevels(x), nlevels(x) - 1))
  ok_sum <- all(abs(colSums(cm)) < tol)
  ok <- ok_dim && ok_sum

  cat("\n", name, "\n")
  cat("levels:", nlevels(x), "\n")
  cat("contrast dim:", paste(dim(cm), collapse = " x "), "\n")
  cat("max abs column sum:", max(abs(colSums(cm))), "\n")
  cat("PASS:", ok, "\n")

  ok
}

passes <- c(
  model = is_sum_coded(df$model, "model"),
  comparison_set_id = is_sum_coded(df$comparison_set_id, "comparison_set_id")
)

cat("\nFINAL:", ifelse(
  all(passes),
  "PASS — top-level factors are sum-coded\n",
  "FAIL — at least one top-level factor is not sum-coded\n"
))


 model 
levels: 18 
contrast dim: 18 x 17 
max abs column sum: 0 
PASS: TRUE 

 comparison_set_id 
levels: 9 
contrast dim: 9 x 8 
max abs column sum: 0 
PASS: TRUE 

FINAL: PASS — top-level factors are sum-coded


In [6]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

check_pair_antisymmetry <- function(dev, set, left, right, label, tol = 1e-8) {
  tmp <- data.frame(
    set = as.character(set),
    left = as.character(left),
    right = as.character(right),
    dev = dev
  )

  tmp$a <- pmin(tmp$left, tmp$right)
  tmp$b <- pmax(tmp$left, tmp$right)

  # Canonical orientation: a -> b.
  # If row is b -> a, flip the sign.
  tmp$canonical_dev <- ifelse(tmp$left == tmp$a, tmp$dev, -tmp$dev)

  pair_check <- aggregate(
    canonical_dev ~ set + a + b,
    data = tmp,
    FUN = function(x) max(x) - min(x)
  )

  names(pair_check)[4] <- "max_diff"

  cat("\n", label, "\n")
  cat("max antisymmetry violation:", max(abs(pair_check$max_diff)), "\n")

  ok <- all(abs(pair_check$max_diff) < tol)

  cat("PASS:", ok, "\n")

  if (!ok) {
    print(pair_check[order(-abs(pair_check$max_diff)), ][1:20, ])
  }

  ok
}

check_within_sum <- function(dev, group, child, label, tol = 1e-8) {
  tmp <- data.frame(group = group, child = child, dev = dev)

  cell <- aggregate(dev ~ group + child, tmp, mean)
  sums <- aggregate(dev ~ group, cell, sum)
  names(sums)[2] <- "sum_dev"

  cat("\n", label, "\n")
  print(sums)

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  ok
}

entity_pair_ok <- check_pair_antisymmetry(
  term_dev(fit, "D_E_within_set"),
  df$comparison_set_id,
  df$left_entity_id,
  df$right_entity_id,
  "pairwise entity-strength differences within comparison_set_id"
)

assay_ok <- check_within_sum(
  term_dev(fit, "A_within_set"),
  df$comparison_set_id,
  df$assay_instance_hash,
  "assay deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  entity_pair_ok && assay_ok,
  "PASS — pairwise entity differences are antisymmetric and assay deviations sum to zero within comparison sets\n",
  "FAIL — at least one pairwise/entity or assay sanity check failed\n"
))


 pairwise entity-strength differences within comparison_set_id 
max antisymmetry violation: 0 
PASS: TRUE 

 assay deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants  1.084202e-18
2        email-providers  0.000000e+00
3  email-providers-china  1.897354e-19
4  email-providers-japan  0.000000e+00
5           model-family  0.000000e+00
6 model-owner-innovation  0.000000e+00
7                   paas  0.000000e+00
8            web-browser -1.734723e-18
9       web-office-tools  0.000000e+00
PASS: TRUE 

FINAL: PASS — pairwise entity differences are antisymmetric and assay deviations sum to zero within comparison sets


In [7]:
%%R

check_model_pair_antisymmetry <- function(dev, model, set, left, right, tol = 1e-8) {
  tmp <- data.frame(
    model = as.character(model),
    set = as.character(set),
    left = as.character(left),
    right = as.character(right),
    dev = dev
  )

  tmp$a <- pmin(tmp$left, tmp$right)
  tmp$b <- pmax(tmp$left, tmp$right)

  # Canonical orientation: a -> b.
  # If row is b -> a, flip the sign.
  tmp$canonical_dev <- ifelse(tmp$left == tmp$a, tmp$dev, -tmp$dev)

  pair_check <- aggregate(
    canonical_dev ~ model + set + a + b,
    data = tmp,
    FUN = function(x) max(x) - min(x)
  )

  names(pair_check)[5] <- "max_diff"

  cat("\nmodel-by-pairwise-entity deviations within comparison_set_id\n")
  cat("max antisymmetry violation:", max(abs(pair_check$max_diff)), "\n")

  ok <- all(abs(pair_check$max_diff) < tol)

  cat("PASS:", ok, "\n")

  if (!ok) {
    print(pair_check[order(-abs(pair_check$max_diff)), ][1:30, ])
  } else {
    print(head(pair_check, 30))
  }

  ok
}

model_entity_pair_ok <- check_model_pair_antisymmetry(
  term_dev(fit, "model:D_E_within_set"),
  df$model,
  df$comparison_set_id,
  df$left_entity_id,
  df$right_entity_id
)


model-by-pairwise-entity deviations within comparison_set_id
max antisymmetry violation: 0 
PASS: TRUE 
                        model                    set           a         b
1             claude-opus-4.6 model-owner-innovation     alibaba anthropic
2           claude-sonnet-4.6 model-owner-innovation     alibaba anthropic
3               deepseek-v3.2 model-owner-innovation     alibaba anthropic
4            gemini-2.5-flash model-owner-innovation     alibaba anthropic
5              gemini-2.5-pro model-owner-innovation     alibaba anthropic
6                 gpt-4o-mini model-owner-innovation     alibaba anthropic
7                     gpt-5.4 model-owner-innovation     alibaba anthropic
8                gpt-oss-120b model-owner-innovation     alibaba anthropic
9                      grok-4 model-owner-innovation     alibaba anthropic
10              grok-4.1-fast model-owner-innovation     alibaba anthropic
11     llama-3.1-70b-instruct model-owner-innovation     alibaba anthr

In [8]:
%%R

X <- model.matrix(fit)

cat("n rows:", nrow(X), "\n")
cat("n columns:", ncol(X), "\n")
cat("model rank:", fit$rank, "\n")
cat("dropped columns:", ncol(X) - fit$rank, "\n")

n rows: 64476 
n columns: 1080 
model rank: 1080 
dropped columns: 0 


In [9]:
%%R

summary(fit)


Call:
lm(formula = score ~ model * comparison_set_id + D_E_within_set + 
    A_within_set + model:D_E_within_set, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.28964 -0.22694  0.01303  0.22497  1.07172 

Coefficients:
                                                                                                                  Estimate
(Intercept)                                                                                                      6.133e-01
model1                                                                                                          -9.248e-02
model2                                                                                                          -3.190e-02
model3                                                                                                           1.056e-01
model4                                                                                                          -1.298e-01
model5           

In [10]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

df$.model_pair_dev <- term_dev(fit, "model:D_E_within_set")

# Entity-perspective version:
# If entity is on the left, keep sign.
# If entity is on the right, flip sign.
left_view <- data.frame(
  model = df$model,
  comparison_set_id = df$comparison_set_id,
  entity_id = df$left_entity_id,
  opponent_entity_id = df$right_entity_id,
  model_entity_pair_dev = df$.model_pair_dev,
  score_from_entity_perspective = df$score
)

right_view <- data.frame(
  model = df$model,
  comparison_set_id = df$comparison_set_id,
  entity_id = df$right_entity_id,
  opponent_entity_id = df$left_entity_id,
  model_entity_pair_dev = -df$.model_pair_dev,
  score_from_entity_perspective = 1 - df$score
)

entity_view <- rbind(left_view, right_view)

model_entity_favouritism <- aggregate(
  cbind(
    model_entity_pair_dev,
    score_from_entity_perspective
  ) ~ model + comparison_set_id + entity_id,
  data = entity_view,
  FUN = mean
)

names(model_entity_favouritism)[
  names(model_entity_favouritism) == "model_entity_pair_dev"
] <- "model_entity_favouritism"

names(model_entity_favouritism)[
  names(model_entity_favouritism) == "score_from_entity_perspective"
] <- "observed_win_rate_from_entity_perspective"

model_entity_favouritism <- model_entity_favouritism[
  order(-model_entity_favouritism$model_entity_favouritism),
]

head(model_entity_favouritism, 30)

                      model      comparison_set_id          entity_id
424           grok-4.1-fast           model-family               grok
790                   phi-4           model-family                phi
974       claude-sonnet-4.6      coding-assistants           windsurf
973         claude-opus-4.6      coding-assistants           windsurf
423                  grok-4           model-family               grok
917    qwen3-235b-a22b-2507           model-family               qwen
1000          grok-4.1-fast model-owner-innovation                xai
935    qwen3-235b-a22b-2507      coding-assistants          qwen-code
516   llama-3.1-8b-instruct model-owner-innovation               meta
625            mistral-nemo model-owner-innovation            mistral
31             mistral-nemo        email-providers            alimail
873                  grok-4  email-providers-china            qq-mail
389  llama-3.1-70b-instruct       web-office-tools   google-workspace
918     qwen3.5-flas